<a href="https://colab.research.google.com/github/thahsinj06/Fly-rank-ml-internship-work/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thahsinj06/Fly-rank-ml-internship-work/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions


### Finding 1 — Content Performance Lifecycle

The paper reports that content performance peaks at 61–90 days,
stabilizes during 91–180 days, and experiences a significant decline
during 271–365 days.

**Methodology question:**

How were the age cohorts constructed, and was the comparison based
on comparable content across the different age ranges? I would want
to understand whether differences in content type, topic, traffic,
or other characteristics could also explain the observed pattern.

This does not dispute the finding; it identifies information that
would help assess how broadly the result can be interpreted.


### Finding 2 — Freshness Multiplier

The paper reports that pages older than 365 days that received an
update within 30 days showed a 3.2× Health Score increase and a 57×
increase in impressions.

**Methodology question:**

Were refreshed pages compared with a comparable group of similarly
aged pages that did not receive a refresh? Such a comparison would
help distinguish the observed association between refreshing and
performance from differences that existed before the refresh.

The result is useful as an observed comparison, but the methodology
would determine how confidently the increase can be attributed to
refreshing itself.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)



### Result

The client-grouped split produced lower Accuracy (0.508 vs 0.557), Precision (0.511 vs 0.558), and F1 (0.648 vs 0.681) compared with the Week-5 random split. Recall remained high and increased slightly from 0.873 to 0.887.

This indicates that the random split gave somewhat more favorable results, particularly for overall accuracy and F1, but the performance did not collapse under the stricter client-grouped validation. The model therefore shows some ability to generalize across previously unseen clients, although the measured performance is lower under the grouped split.

These results are directional rather than proof of real-world performance. A larger set of client-held-out or time-aware evaluations would provide stronger evidence of generalization.



In [10]:
# ============================================
# SECTION 2 — HONEST SPLIT: BEFORE vs AFTER
# ============================================

import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# --------------------------------------------
# 1. Load dataset
# --------------------------------------------

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "thahsinj06/Fly-rank-ml-internship-work/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)


# --------------------------------------------
# 2. Create target
# --------------------------------------------

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)


# --------------------------------------------
# 3. Define Week-5 feature set
# --------------------------------------------

features = [
    "search_volume",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr"
]

X = df[features]
y = df["is_declining_label"]

print("\nFeatures used:")
print(features)

print("\nTarget distribution:")
print(y.value_counts())

print("\nMissing values in features:")
print(X.isna().sum())


# ============================================
# BEFORE — Week-5 random split
# ============================================

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Imputation + scaling + Logistic Regression
model_random = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model_random.fit(X_train_random, y_train_random)

y_pred_random = model_random.predict(X_test_random)

before_metrics = {
    "Accuracy": accuracy_score(y_test_random, y_pred_random),
    "Precision": precision_score(y_test_random, y_pred_random),
    "Recall": recall_score(y_test_random, y_pred_random),
    "F1": f1_score(y_test_random, y_pred_random)
}


# ============================================
# AFTER — Client-grouped split
# ============================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=df["client_id"])
)

X_train_grouped = X.iloc[train_idx]
X_test_grouped = X.iloc[test_idx]

y_train_grouped = y.iloc[train_idx]
y_test_grouped = y.iloc[test_idx]

# Same preprocessing + same model
model_grouped = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model_grouped.fit(X_train_grouped, y_train_grouped)

y_pred_grouped = model_grouped.predict(X_test_grouped)

after_metrics = {
    "Accuracy": accuracy_score(y_test_grouped, y_pred_grouped),
    "Precision": precision_score(y_test_grouped, y_pred_grouped),
    "Recall": recall_score(y_test_grouped, y_pred_grouped),
    "F1": f1_score(y_test_grouped, y_pred_grouped)
}


# ============================================
# BEFORE vs AFTER COMPARISON
# ============================================

comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1"],
    "Week-5 Random Split": [
        before_metrics["Accuracy"],
        before_metrics["Precision"],
        before_metrics["Recall"],
        before_metrics["F1"]
    ],
    "Client-Grouped Split": [
        after_metrics["Accuracy"],
        after_metrics["Precision"],
        after_metrics["Recall"],
        after_metrics["F1"]
    ]
})

comparison["Change"] = (
    comparison["Client-Grouped Split"]
    - comparison["Week-5 Random Split"]
)

print("\nBEFORE vs AFTER")
print("=" * 60)

display(comparison.round(3))


# ============================================
# CLIENT SEPARATION CHECK
# ============================================

train_clients = set(
    df.iloc[train_idx]["client_id"]
)

test_clients = set(
    df.iloc[test_idx]["client_id"]
)

client_overlap = train_clients.intersection(test_clients)

print("\nCLIENT-GROUPING CHECK")
print("=" * 60)
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Client overlap:", len(client_overlap))

if len(client_overlap) == 0:
    print("PASS — No client appears in both training and test sets.")
else:
    print("WARNING — Client overlap detected.")

Dataset loaded successfully.
Dataset shape: (30000, 44)

Features used:
['search_volume', 'impressions_90d', 'days_since_last_update', 'avg_position', 'ctr']

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Missing values in features:
search_volume             2468
impressions_90d              0
days_since_last_update       0
avg_position                 0
ctr                          0
dtype: int64

BEFORE vs AFTER


,Metric,Week-5 Random Split,Client-Grouped Split,Change
0,Accuracy,0.557,0.508,-0.049
1,Precision,0.558,0.511,-0.048
2,Recall,0.873,0.887,0.015
3,F1,0.681,0.648,-0.033



CLIENT-GROUPING CHECK
Training clients: 25
Testing clients: 7
Client overlap: 0
PASS — No client appears in both training and test sets.


## 3. Leakage audit

### Leakage audit result

The final feature set does not contain the target columns `trend_direction` or `trend_pct`, so there is no direct target leakage. However, `trend_direction` was used to create the target, and some performance features may overlap with the period used to define that trend. Therefore, temporal leakage cannot be ruled out without knowing the exact target-generation window.


In [11]:
# ============================================
# SECTION 3 — LEAKAGE AUDIT
# ============================================

print("LEAKAGE AUDIT")
print("=" * 50)

# Final model features
final_features = [
    "search_volume",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr"
]

# 1. Direct target leakage check
target_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

direct_leakage = [
    col for col in final_features
    if col in target_columns
]

print("\n1. Direct target leakage:")
print("Suspicious features:", direct_leakage if direct_leakage else "None")


# 2. Check missing values
print("\n2. Missing values:")
print(df[final_features].isna().sum())


# 3. Check feature-target correlations
print("\n3. Feature-target correlations:")

correlations = (
    df[final_features + ["is_declining_label"]]
    .corr()["is_declining_label"]
    .drop("is_declining_label")
    .sort_values(key=abs, ascending=False)
)

print(correlations.round(3))


# 4. Check target construction
print("\n4. Target construction:")
print("Target = (trend_direction == 'down')")
print("trend_direction used as feature:", "trend_direction" in final_features)
print("trend_pct used as feature:", "trend_pct" in final_features)


# 5. Final audit
print("\n5. AUDIT CONCLUSION:")

if not direct_leakage:
    print("PASS — No direct target columns are present in the final feature set.")
else:
    print("WARNING — Direct target leakage detected.")

print(
    "\nTemporal overlap should still be checked because "
    "the exact window used to calculate trend_direction is not known."
)

LEAKAGE AUDIT

1. Direct target leakage:
Suspicious features: None

2. Missing values:
search_volume             2468
impressions_90d              0
days_since_last_update       0
avg_position                 0
ctr                          0
dtype: int64

3. Feature-target correlations:
days_since_last_update    0.081
ctr                      -0.062
avg_position             -0.029
search_volume            -0.019
impressions_90d          -0.018
Name: is_declining_label, dtype: float64

4. Target construction:
Target = (trend_direction == 'down')
trend_direction used as feature: False
trend_pct used as feature: False

5. AUDIT CONCLUSION:
PASS — No direct target columns are present in the final feature set.

Temporal overlap should still be checked because the exact window used to calculate trend_direction is not known.


## 4. Claim rewrite



**Original claim:**
The model accurately predicts which content is declining and can be used to identify pages that need refreshing.

**Safer claim:**
The Logistic Regression model **measured** an F1 score of 0.681 on the Week-5 random split and 0.648 on the client-grouped split. The results provide **directional decision-support** for prioritizing potentially declining content, but they do not establish causation or guarantee real-world performance.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.